In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm
import itertools

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx

In [2]:
import sys
import os

sys.path.append(os.path.abspath("../src"))

In [3]:
from true_graph import TrueGraph
from learner import FactorGraphLearner
from noise_generator import IndependentMarginals
from metrics import *
from random_graph import generate_random_graph, generate_random_tree
from chow_liu import chow_liu

# Trees

In [12]:
TREE_SIZES = np.arange(4,61,4, dtype=np.int32)

In [13]:
tree_list = []
sample_list = []

for n in tqdm(TREE_SIZES):
    rng = np.random.default_rng(seed=165)
    g = generate_random_tree(n, 2, rng)
    samps = g.sample(500, seed=650, progress=False)
    tree_list.append(g)
    sample_list.append(samps)

100%|███████████████████████████████████████████| 15/15 [01:10<00:00,  4.71s/it]


# Graphs

In [16]:
N_VARIABLE = 10
NUM_FACTORS = list(range(4,12))

graph_list = []
graph_sample_list = []

for nf in tqdm(NUM_FACTORS):
    rng = np.random.default_rng(seed=882)
    g = generate_random_graph(10, 2, rng, n_factors=nf)
    samps = g.sample(500, seed=648, progress=False)
    graph_list.append(g)
    graph_sample_list.append(samps)

100%|█████████████████████████████████████████████| 8/8 [00:12<00:00,  1.54s/it]


# Learn

In [22]:
def run_learners(graph_list, sample_list, 
                 K_mult=1.5, epochs=100, lr=0.01, seed=123,
                 lambda_bp=0.005, lambda_mask=1e-4, lambda_weight=1e-4, lambda_mlp=1e-2, schedule=True):
    results = []
    for i in tqdm(range(len(graph_list))):
        graph = graph_list[i]
        samples = sample_list[i]
        n_vars = graph.n
        alph_size = graph.alphabet_sizes[0]
        n_factors = len(graph.factors)
        n_poss = alph_size**n_vars
        ng = IndependentMarginals(samples, alphabet_size=alph_size, alpha=1, seed=714)
        lrn = FactorGraphLearner(
            n_vars=n_vars,
            alphabet_size=alph_size,
            K=int(K_mult*n_factors),
            noise_generator=ng,
            hidden_dims=(16,16),
            max_factor_size=5,
            shared_mlp=False,
            seed=seed
        )
        lrn.train(
            samples,
            n_epochs=epochs,
            lr=lr,
            lambda_mask=lambda_mask,
            lambda_weight=lambda_weight,
            lambda_mlp_l2=lambda_mlp,
            lambda_bp=lambda_bp,
            penalty_schedule=schedule,
            verbose=False
        )
        chow_liu_res = chow_liu(samples, alph_size)
        results.append({
            'learner' : lrn,
            'n_variables' : n_vars,
            'n_factors' : n_factors,
            'KL' : kl_divergence(graph, lrn) if n_poss <= 100000 else None,
            'KL Chow-Liu' : kl_divergence(graph, chow_liu_res) if n_poss <= 100000 else None,
            'KL MLE' : kl_mle_optimal(graph, samples) if n_poss <= 100000 else None,
            'SHD' : structural_hamming_distance(graph, lrn),
            'SHD Chow-Liu' : structural_hamming_distance(graph, chow_liu_res),
            'true density' : graph_density(graph),
            'density' : graph_density(lrn)
        })
    return results

In [23]:
results_trees = run_learners(tree_list, sample_list)

100%|███████████████████████████████████████████| 15/15 [03:21<00:00, 13.40s/it]


In [24]:
results_graphs = run_learners(graph_list, graph_sample_list)

100%|█████████████████████████████████████████████| 8/8 [00:26<00:00,  3.31s/it]


# Visuals

In [25]:
# KL / SHD vs n_vars or true graph density/number_factors